# 00 — Simplified all-in-one pipeline
Один независимый ноутбук: данные → идентификация → epsilon → Lyapunov → feedback.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import ensure_dir, set_seed, save_dataframe, save_json, get_repo_root
from src.systems import cross_coupled_uncontrolled
from src.simulation import simulate_batch, uncertain_dynamics
from src.plots import set_plot_style, save_figure, plot_phase_trajectories, plot_residual_histogram, plot_lyapunov_contours
from src.basis import get_basis
from src.identification import fit_identified_model, predict_vector_field, rmse
from src.uncertainty import compute_residuals, residual_norms, estimate_epsilon, compute_bounding_box, bounded_disturbance
from src.lyapunov import numerical_jacobian, solve_lyapunov, evaluate_lyapunov_grid
from src.control import candidate_gains, closed_loop_jacobian

ROOT = get_repo_root()
DATA_DIR = ROOT / 'data' / 'processed'
FIG_DIR = ROOT / 'results' / 'figures'
METRICS_DIR = ROOT / 'results' / 'metrics'
TABLES_DIR = ROOT / 'results' / 'tables'
for p in [DATA_DIR, FIG_DIR, METRICS_DIR, TABLES_DIR]:
    ensure_dir(p)


In [ ]:
set_seed(42)
set_plot_style()
# 1) Data generation
t_eval = np.linspace(0, 10, 400)
initials = np.array([[-1.2, -0.8], [-1.0, 0.7], [-0.6, 1.1], [0.5, -1.0], [1.0, 0.9], [1.3, -0.4]], dtype=float)
traj_true = simulate_batch(cross_coupled_uncontrolled, initials, (0, 10), t_eval)
X = np.vstack([states for _, states in traj_true])
Xdot = np.vstack([np.array([cross_coupled_uncontrolled(float(t), s) for t, s in zip(ts, states)]) for ts, states in traj_true])
save_dataframe(pd.DataFrame(np.hstack([X, Xdot]), columns=['x1', 'x2', 'xdot1', 'xdot2']), DATA_DIR / 'cross_coupled_dataset.csv')

# 2) Identification
basis_fn = get_basis('quadratic_with_constant')
model = fit_identified_model(X, Xdot, basis_fn=basis_fn, basis_name='quadratic_with_constant')
Xdot_hat = predict_vector_field(X, basis_fn, model.coefficients)
res = compute_residuals(Xdot, Xdot_hat)
eps = estimate_epsilon(residual_norms(res), q=0.95)

# 3) Open-loop Lyapunov and uncertain simulation
fhat = lambda t, x: predict_vector_field(x[None, :], basis_fn, model.coefficients)[0]
A = numerical_jacobian(lambda x: fhat(0.0, x), np.zeros(2))
P = solve_lyapunov(A, np.eye(2))
unc = uncertain_dynamics(fhat, lambda t: bounded_disturbance(t, eps))
traj_unc = simulate_batch(unc, np.array([[-1.0,-0.8],[-0.8,1.0],[0.8,-1.0],[1.1,0.9]]), (0, 12), np.linspace(0, 12, 500))

# 4) Controlled uncertain simulation
B = np.array([[0.0], [1.0]])
K = candidate_gains()['medium']
cl = lambda t, x: cross_coupled_uncontrolled(t, x) + (B @ (K @ x)).reshape(-1) + bounded_disturbance(t, eps)
traj_cl = simulate_batch(cl, np.array([[-1.0,-0.8],[-0.8,1.0],[0.8,-1.0],[1.1,0.9]]), (0, 12), np.linspace(0, 12, 500))

# 5) Save compact set of figures and metrics
plt.figure(); plot_phase_trajectories(traj_true, 'True system'); save_figure(FIG_DIR / 'phase_true.png'); plt.show()
plt.figure(); plot_phase_trajectories(traj_unc, 'Uncertain uncontrolled'); save_figure(FIG_DIR / 'uncertain_uncontrolled.png'); plt.show()
plt.figure(); plot_phase_trajectories(traj_cl, 'Uncertain controlled (K=medium)'); save_figure(FIG_DIR / 'uncertain_controlled_medium.png'); plt.show()

xx, yy, vv = evaluate_lyapunov_grid(P, (-1.5, 1.5), (-1.5, 1.5), points=80)
plt.figure(); plot_lyapunov_contours(xx, yy, vv); save_figure(FIG_DIR / 'lyapunov_level_sets.png'); plt.show()

save_json({'rmse': rmse(Xdot, Xdot_hat), 'epsilon_q95': float(eps), 'A': A.tolist(), 'P': P.tolist()}, METRICS_DIR / 'all_in_one_metrics.json')
print('Done. RMSE=', rmse(Xdot, Xdot_hat), 'epsilon=', eps)
